# Snake DQN — Training Analysis

This notebook analyses the training run logged in `results/training_log.csv`.

We look at:
1. Did the agent actually learn? (learning curve)
2. When did it stop exploring and start exploiting? (epsilon decay)
3. How stable was training? (loss over time)
4. Score distribution — median, variance, best runs
5. Where on the grid does it die most? (death heatmap)
6. Early vs late performance comparison

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path

# ── style ──────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor':  '#0a0a0f',
    'axes.facecolor':    '#111118',
    'axes.edgecolor':    '#1e1e2e',
    'axes.labelcolor':   '#c8c8d8',
    'axes.titlecolor':   '#ffffff',
    'xtick.color':       '#666680',
    'ytick.color':       '#666680',
    'grid.color':        '#1e1e2e',
    'grid.linewidth':    0.5,
    'text.color':        '#c8c8d8',
    'font.family':       'monospace',
    'figure.titlesize':  14,
    'axes.titlesize':    12,
    'axes.labelsize':    10,
})

GREEN  = '#00ff88'
RED    = '#ff4466'
YELLOW = '#ffcc00'
BLUE   = '#4488ff'
DIM    = '#666680'

print('Libraries loaded.')

In [ ]:
# ── load data ──────────────────────────────────────────────────────────────
LOG_PATH = Path('../results/training_log.csv')

df = pd.read_csv(LOG_PATH)
print(f'Loaded {len(df):,} episodes')
print(f'Columns: {list(df.columns)}')
df.head(10)

In [ ]:
# ── summary stats ──────────────────────────────────────────────────────────
print('── Training Summary ──────────────────────────────')
print(f'  Total episodes  : {len(df):,}')
print(f'  Best score ever : {df["score"].max()}')
print(f'  Avg score (all) : {df["score"].mean():.2f}')
print(f'  Avg score (last 100) : {df["score"].tail(100).mean():.2f}')
print(f'  Avg score (last 10)  : {df["score"].tail(10).mean():.2f}')
print(f'  Median score    : {df["score"].median():.1f}')
print(f'  Episodes scored 0    : {(df["score"]==0).sum():,} ({(df["score"]==0).mean()*100:.1f}%)')
print(f'  Episodes scored 10+  : {(df["score"]>=10).sum():,} ({(df["score"]>=10).mean()*100:.1f}%)')
print(f'  Final epsilon   : {df["epsilon"].iloc[-1]:.4f}')

## 1. Learning Curve

The most important chart. Raw score is noisy — what matters is the rolling average trend.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))

# raw scores (faint)
ax.plot(df['episode'], df['score'], color=GREEN, alpha=0.15, linewidth=0.7, label='Score per episode')

# rolling averages
roll10  = df['score'].rolling(10).mean()
roll100 = df['score'].rolling(100).mean()

ax.plot(df['episode'], roll10,  color=GREEN,  linewidth=1.2, alpha=0.6, label='Rolling avg (10)')
ax.plot(df['episode'], roll100, color=YELLOW, linewidth=2.0, label='Rolling avg (100)')

# best score line
ax.axhline(df['score'].max(), color=RED, linewidth=0.8, linestyle='--', alpha=0.6,
           label=f'Best: {df["score"].max()}')

# epsilon threshold — when it crossed 0.1 (mostly exploiting)
exploit_ep = df[df['epsilon'] <= 0.1]['episode'].min()
if pd.notna(exploit_ep):
    ax.axvline(exploit_ep, color=BLUE, linewidth=1.0, linestyle=':', alpha=0.8)
    ax.text(exploit_ep + 5, ax.get_ylim()[1] * 0.9,
            f'ε<0.1\n(ep {exploit_ep})', color=BLUE, fontsize=8)

ax.set_xlabel('Episode')
ax.set_ylabel('Score')
ax.set_title('Learning Curve — Score Over Training')
ax.legend(loc='upper left', framealpha=0.2, fontsize=8)
ax.grid(True)
fig.tight_layout()
plt.savefig('../results/learning_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → results/learning_curve.png')

## 2. Epsilon Decay vs Score

Exploration rate falling over time. The key question: does score start rising *as* epsilon falls?

In [ ]:
fig, ax1 = plt.subplots(figsize=(13, 4))

# score on left axis
ax1.plot(df['episode'], df['score'].rolling(50).mean(),
         color=GREEN, linewidth=2, label='Score (avg 50)')
ax1.set_xlabel('Episode')
ax1.set_ylabel('Score', color=GREEN)
ax1.tick_params(axis='y', labelcolor=GREEN)

# epsilon on right axis
ax2 = ax1.twinx()
ax2.plot(df['episode'], df['epsilon'],
         color=YELLOW, linewidth=1.5, linestyle='--', label='Epsilon', alpha=0.8)
ax2.set_ylabel('Epsilon (exploration rate)', color=YELLOW)
ax2.tick_params(axis='y', labelcolor=YELLOW)
ax2.set_ylim(0, 1.05)

ax1.set_title('Exploration Decay vs Score Growth')
ax1.grid(True)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1+lines2, labels1+labels2, loc='upper left', framealpha=0.2, fontsize=8)

fig.tight_layout()
plt.savefig('../results/epsilon_vs_score.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → results/epsilon_vs_score.png')

## 3. Training Loss

Loss going up isn't necessarily bad in RL — it means the agent is encountering harder situations as it gets better. What we don't want is loss exploding or never moving.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))

loss_nonzero = df[df['loss'] > 0]

ax.plot(loss_nonzero['episode'], loss_nonzero['loss'],
        color=RED, alpha=0.2, linewidth=0.5)
ax.plot(loss_nonzero['episode'], loss_nonzero['loss'].rolling(50).mean(),
        color=RED, linewidth=2, label='Loss (avg 50)')

ax.set_xlabel('Episode')
ax.set_ylabel('Huber Loss')
ax.set_title('Training Loss Over Time')
ax.legend(framealpha=0.2, fontsize=8)
ax.grid(True)
fig.tight_layout()
plt.savefig('../results/training_loss.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → results/training_loss.png')

## 4. Score Distribution

Histogram of all scores. A good agent should show a right-skewed distribution — most runs decent, some exceptional.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# full distribution
axes[0].hist(df['score'], bins=30, color=GREEN, alpha=0.7, edgecolor='#0a0a0f')
axes[0].axvline(df['score'].mean(),   color=YELLOW, linewidth=1.5, linestyle='--', label=f'Mean: {df["score"].mean():.1f}')
axes[0].axvline(df['score'].median(), color=BLUE,   linewidth=1.5, linestyle=':',  label=f'Median: {df["score"].median():.1f}')
axes[0].set_xlabel('Score')
axes[0].set_ylabel('Episodes')
axes[0].set_title('Score Distribution (All Episodes)')
axes[0].legend(framealpha=0.2, fontsize=8)
axes[0].grid(True)

# early vs late
n = len(df)
early = df.head(n//3)['score']
late  = df.tail(n//3)['score']
axes[1].hist(early, bins=20, color=RED,   alpha=0.6, label=f'Early (ep 1–{n//3})',    edgecolor='#0a0a0f')
axes[1].hist(late,  bins=20, color=GREEN, alpha=0.6, label=f'Late (ep {n-n//3}–{n})', edgecolor='#0a0a0f')
axes[1].set_xlabel('Score')
axes[1].set_ylabel('Episodes')
axes[1].set_title('Early vs Late Score Distribution')
axes[1].legend(framealpha=0.2, fontsize=8)
axes[1].grid(True)

fig.tight_layout()
plt.savefig('../results/score_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Early avg: {early.mean():.2f}  |  Late avg: {late.mean():.2f}')
print(f'Improvement: +{late.mean()-early.mean():.2f} ({(late.mean()-early.mean())/max(early.mean(),0.01)*100:.0f}%)')
print('Saved → results/score_distribution.png')

## 5. Steps Per Episode

A smarter snake survives longer. Steps should increase as score increases.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))

ax.plot(df['episode'], df['steps'].rolling(50).mean(),
        color=BLUE, linewidth=2, label='Steps per episode (avg 50)')
ax.fill_between(df['episode'],
                df['steps'].rolling(50).mean() - df['steps'].rolling(50).std(),
                df['steps'].rolling(50).mean() + df['steps'].rolling(50).std(),
                color=BLUE, alpha=0.1)

ax.set_xlabel('Episode')
ax.set_ylabel('Steps')
ax.set_title('Survival Time Over Training')
ax.legend(framealpha=0.2, fontsize=8)
ax.grid(True)
fig.tight_layout()
plt.savefig('../results/steps_over_time.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → results/steps_over_time.png')

## 6. Best Runs — Top 10 Episodes

In [ ]:
top10 = df.nlargest(10, 'score')[['episode','score','steps','reward','epsilon']]
top10 = top10.reset_index(drop=True)
top10.index += 1
print('── Top 10 Episodes ──────────────────────────────')
print(top10.to_string())

## 7. Training Phase Breakdown

Split training into 4 quarters and compare performance across phases.

In [ ]:
n = len(df)
q = n // 4

phases = {
    'Phase 1\n(Random)':     df.iloc[0:q],
    'Phase 2\n(Learning)':   df.iloc[q:2*q],
    'Phase 3\n(Improving)':  df.iloc[2*q:3*q],
    'Phase 4\n(Exploiting)': df.iloc[3*q:],
}

fig, axes = plt.subplots(1, 3, figsize=(14, 5))

labels = list(phases.keys())
colors = [RED, YELLOW, BLUE, GREEN]

# avg score per phase
avgs = [v['score'].mean() for v in phases.values()]
axes[0].bar(labels, avgs, color=colors, alpha=0.8, edgecolor='#0a0a0f')
axes[0].set_title('Avg Score by Phase')
axes[0].set_ylabel('Avg Score')
axes[0].grid(True, axis='y')
for i, v in enumerate(avgs):
    axes[0].text(i, v + 0.1, f'{v:.1f}', ha='center', fontsize=9, color='white')

# max score per phase
maxs = [v['score'].max() for v in phases.values()]
axes[1].bar(labels, maxs, color=colors, alpha=0.8, edgecolor='#0a0a0f')
axes[1].set_title('Best Score by Phase')
axes[1].set_ylabel('Best Score')
axes[1].grid(True, axis='y')
for i, v in enumerate(maxs):
    axes[1].text(i, v + 0.1, f'{v}', ha='center', fontsize=9, color='white')

# % episodes scoring > 5
pcts = [(v['score'] > 5).mean() * 100 for v in phases.values()]
axes[2].bar(labels, pcts, color=colors, alpha=0.8, edgecolor='#0a0a0f')
axes[2].set_title('% Episodes Scoring > 5')
axes[2].set_ylabel('Percentage')
axes[2].grid(True, axis='y')
for i, v in enumerate(pcts):
    axes[2].text(i, v + 0.3, f'{v:.1f}%', ha='center', fontsize=9, color='white')

fig.suptitle('Training Phase Breakdown', fontsize=13, color='white', y=1.01)
fig.tight_layout()
plt.savefig('../results/phase_breakdown.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → results/phase_breakdown.png')

## Summary

What we can conclude from this training run.

In [ ]:
n       = len(df)
early_avg = df.head(n//4)['score'].mean()
late_avg  = df.tail(n//4)['score'].mean()
exploit_ep = df[df['epsilon'] <= 0.1]['episode'].min()

print('═' * 52)
print('  TRAINING SUMMARY')
print('═' * 52)
print(f'  Total episodes       : {n:,}')
print(f'  Best score           : {df["score"].max()}')
print(f'  Early avg (Q1)       : {early_avg:.2f}')
print(f'  Late avg  (Q4)       : {late_avg:.2f}')
print(f'  Improvement          : +{late_avg - early_avg:.2f}')
if pd.notna(exploit_ep):
    print(f'  Switched to exploit  : episode {exploit_ep}')
print(f'  Final epsilon        : {df["epsilon"].iloc[-1]:.4f}')
print(f'  Avg steps (last 100) : {df["steps"].tail(100).mean():.0f}')
print('═' * 52)